# Assignment 5: Build and Evaluate Classification Models

Juan Maldonado Franco  
DDS-8555 Predictive Analysis  
Mohamed Nabeel

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / "DDS-8555 - Predictive Analysis").exists():
        COURSE = parent / "DDS-8555 - Predictive Analysis"
        break
else:
    COURSE = ROOT.parents[1]
DATA = COURSE / "data"
KAGGLE = DATA / "kaggle"
SUBMISSIONS = DATA / "submissions"
RANDOM_STATE = 42
pd.set_option("display.max_columns", 80)

## Conceptual Question 1

The logistic model can be written as p(X) = exp(beta0 + beta1X) / [1 + exp(beta0 + beta1X)].  Dividing p(X) by 1 - p(X) cancels the denominator and leaves exp(beta0 + beta1X).  Taking the natural log gives log[p(X)/(1 - p(X))] = beta0 + beta1X.  This proves that the probability form and log-odds form are the same model written on different scales.

## Applied Question 13: Weekly Classification

The Weekly data exercise compares logistic regression, LDA, QDA, KNN, and naive Bayes on held-out stock direction data.  The main issue is not just accuracy, but the type of errors each model makes.

In [2]:
from ISLP import load_data
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

weekly = load_data("Weekly")
display(weekly.describe())
train = weekly["Year"] <= 2008
test = ~train
X_train = weekly.loc[train, ["Lag2"]]
X_test = weekly.loc[test, ["Lag2"]]
y_train = weekly.loc[train, "Direction"]
y_test = weekly.loc[test, "Direction"]
models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(),
    "KNN-1": Pipeline([("scale", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=1))]),
    "Naive Bayes": GaussianNB(),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    })
display(pd.DataFrame(rows).sort_values("accuracy", ascending=False))

,Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today
count,1089.000000,1089.000000,1089.000000,1089.000000,1089.000000,1089.000000,1089.000000,1089.000000
mean,2000.048669,0.150585,0.151079,0.147205,0.145818,0.139893,1.574618,0.149899
std,6.033182,2.357013,2.357254,2.360502,2.360279,2.361285,1.686636,2.356927
min,1990.000000,-18.195000,-18.195000,-18.195000,-18.195000,-18.195000,0.087465,-18.195000
25%,1995.000000,-1.154000,-1.154000,-1.158000,-1.158000,-1.166000,0.332022,-1.154000
50%,2000.000000,0.241000,0.241000,0.241000,0.238000,0.234000,1.002680,0.241000
75%,2005.000000,1.405000,1.409000,1.409000,1.409000,1.405000,2.053727,1.405000
max,2010.000000,12.026000,12.026000,12.026000,12.026000,12.026000,9.328214,12.026000


,model,accuracy,balanced_accuracy,confusion_matrix
0,Logistic,0.625000,0.563668,"[[9, 34], [5, 56]]"
1,LDA,0.625000,0.563668,"[[9, 34], [5, 56]]"
2,QDA,0.586538,0.500000,"[[0, 43], [0, 61]]"
4,Naive Bayes,0.586538,0.500000,"[[0, 43], [0, 61]]"
3,KNN-1,0.490385,0.493519,"[[22, 21], [32, 29]]"


## Kaggle Obesity Classification

The competition required multinomial logistic regression, LDA or QDA, naive Bayes, and SVM submissions.  These models represent different assumptions about decision boundaries and feature distributions.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

obesity = pd.read_csv(KAGGLE / "playground-series-s4e2" / "train.csv")
X = obesity.drop(columns=["NObeyesdad"])
y = obesity["NObeyesdad"]
display(y.value_counts(normalize=True).rename("class_share").to_frame())
cat = X.select_dtypes(include="object").columns.tolist()
num = [c for c in X.columns if c not in cat + ["id"]]
pre_dense = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat), ("num", StandardScaler(), num)], sparse_threshold=0)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=.2, stratify=y, random_state=RANDOM_STATE)
models = {
    "Multinomial logistic": LogisticRegression(max_iter=2000, C=1.0),
    "LDA": LinearDiscriminantAnalysis(),
    "Naive Bayes": GaussianNB(),
    "Linear SVM": LinearSVC(C=.5, random_state=RANDOM_STATE),
}
rows = []
fitted_models = {}
for name, clf in models.items():
    model = Pipeline([("pre", pre_dense), ("model", clf)])
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    fitted_models[name] = (model, pred)
    rows.append({
        "model": name,
        "validation_accuracy": accuracy_score(y_valid, pred),
        "balanced_accuracy": balanced_accuracy_score(y_valid, pred),
    })
validation = pd.DataFrame(rows).sort_values("validation_accuracy", ascending=False)
display(validation)
display(pd.DataFrame({
    "model": ["Multinomial logistic", "LDA", "Naive Bayes", "Linear SVM"],
    "main_assumption_or_risk": [
        "linear class boundaries on transformed predictors; regularization controls instability",
        "approximately normal class-conditional predictors with shared covariance",
        "conditional independence of predictors within each class",
        "separable margin after preprocessing; hinge-loss model can be sensitive to scaling",
    ],
}))
best_name = validation.iloc[0]["model"]
best_model, best_pred = fitted_models[best_name]
display(pd.DataFrame(confusion_matrix(y_valid, best_pred, labels=best_model.classes_), index=best_model.classes_, columns=best_model.classes_))

,class_share
NObeyesdad,
Obesity_Type_III,0.194913
Obesity_Type_II,0.156470
Normal_Weight,0.148473
Obesity_Type_I,0.140187
Insufficient_Weight,0.121544
Overweight_Level_II,0.121495
Overweight_Level_I,0.116919


,model,validation_accuracy,balanced_accuracy
0,Multinomial logistic,0.868738,0.854949
1,LDA,0.822977,0.806228
3,Linear SVM,0.748555,0.719493
2,Naive Bayes,0.585983,0.549357


,model,main_assumption_or_risk
0,Multinomial logistic,linear class boundaries on transformed predict...
1,LDA,approximately normal class-conditional predict...
2,Naive Bayes,conditional independence of predictors within ...
3,Linear SVM,separable margin after preprocessing; hinge-lo...


,Insufficient_Weight,Normal_Weight,Obesity_Type_I,Obesity_Type_II,Obesity_Type_III,Overweight_Level_I,Overweight_Level_II
Insufficient_Weight,479,25,0,0,0,1,0
Normal_Weight,58,505,2,0,0,43,9
Obesity_Type_I,1,0,493,36,3,11,38
Obesity_Type_II,0,0,22,625,0,0,3
Obesity_Type_III,0,0,0,1,807,1,0
Overweight_Level_I,1,46,15,0,0,342,81
Overweight_Level_II,0,3,79,8,0,58,356


In [4]:
import re

status_path = SUBMISSIONS / "kaggle_submission_status_playground-series-s4e2.txt"
for encoding in ("utf-16", "utf-8"):
    try:
        text = status_path.read_text(encoding=encoding)
        break
    except UnicodeError:
        continue

records = []
for line in text.splitlines():
    parts = re.split(r"\s{2,}", line.strip())
    if len(parts) >= 7 and parts[0].isdigit() and "A5_" in parts[1]:
        records.append({
            "ref": parts[0],
            "fileName": parts[1],
            "date": parts[2],
            "description": parts[3],
            "status": parts[4],
            "publicScore": parts[5],
            "privateScore": parts[6],
        })

display(pd.DataFrame(records))
display(pd.DataFrame({
    "submission_file": ['A5_multinomial_logistic_playground_series_s4e2.csv', 'A5_lda_playground_series_s4e2.csv', 'A5_naive_bayes_playground_series_s4e2.csv', 'A5_svm_playground_series_s4e2.csv'],
    "exists_locally": [(SUBMISSIONS / file).exists() for file in ['A5_multinomial_logistic_playground_series_s4e2.csv', 'A5_lda_playground_series_s4e2.csv', 'A5_naive_bayes_playground_series_s4e2.csv', 'A5_svm_playground_series_s4e2.csv']],
}))
display(pd.DataFrame({
    "evidence": ["Public GitHub repository", "Notebook path in repository"],
    "value": [
        "https://github.com/maldo81/dds-8555-predictive-analysis",
        "Week 5/Assignment 5/MaldonadoJDDS8555-5.ipynb",
    ],
}))

,ref,fileName,date,description,status,publicScore,privateScore
0,52966803,A5_svm_playground_series_s4e2.csv,2026-05-23 21:27:31.870000,DDS-8555 A5 SVM model,SubmissionStatus.COMPLETE,0.87789,0.87888
1,52966801,A5_naive_bayes_playground_series_s4e2.csv,2026-05-23 21:27:29.533000,DDS-8555 A5 naive Bayes model,SubmissionStatus.COMPLETE,0.58995,0.58544
2,52966800,A5_lda_playground_series_s4e2.csv,2026-05-23 21:27:27.093000,DDS-8555 A5 LDA model,SubmissionStatus.COMPLETE,0.82153,0.82108
3,52966799,A5_multinomial_logistic_playground_series_s4e2...,2026-05-23 21:27:24.603000,DDS-8555 A5 multinomial logistic regression,SubmissionStatus.COMPLETE,0.86632,0.86497


,submission_file,exists_locally
0,A5_multinomial_logistic_playground_series_s4e2...,True
1,A5_lda_playground_series_s4e2.csv,True
2,A5_naive_bayes_playground_series_s4e2.csv,True
3,A5_svm_playground_series_s4e2.csv,True


,evidence,value
0,Public GitHub repository,https://github.com/maldo81/dds-8555-predictive...
1,Notebook path in repository,Week 5/Assignment 5/MaldonadoJDDS8555-5.ipynb


## Interpretation

The SVM and multinomial logistic models were stronger than naive Bayes on the Obesity data because the predictors are not conditionally independent in a realistic health-behavior data set.  LDA was competitive but more assumption-bound because it relies on class-conditional normality and shared covariance structure.  The assumption table makes the model risks visible, and the confusion matrix for the best validation model shows which obesity categories are most likely to be confused.  The Kaggle evidence confirms all four required submissions were completed.  Because obesity classification is tied to population-level body-mass trends, the predictions should be interpreted as competition labels rather than clinical diagnoses (NCD Risk Factor Collaboration, 2016).

## References

Cortes, C., & Vapnik, V. (1995).  Support-vector networks. *Machine Learning, 20*, 273-297. https://doi.org/10.1007/BF00994018

James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An introduction to statistical learning: With applications in Python*.  Springer. https://doi.org/10.1007/978-3-031-38747-0

NCD Risk Factor Collaboration. (2016).  Trends in adult body-mass index in 200 countries from 1975 to 2014: A pooled analysis of 1698 population-based measurement studies with 19.2 million participants. *The Lancet, 387*(10026), 1377-1396. https://doi.org/10.1016/S0140-6736(16)30054-X

Sokolova, M., & Lapalme, G. (2009).  A systematic analysis of performance measures for classification tasks. *Information Processing & Management, 45*(4), 427-437. https://doi.org/10.1016/j.ipm.2009.03.002